In [1]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch.nn as nn
from transformers import AdamW
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
from imblearn.under_sampling import RandomUnderSampler
import os
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [32]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
# df = pd.read_csv('../data/df_sentences_with_gender.csv')
df = pd.read_csv('../data/df_sentences_without_gender.csv')

# Process Data Into Sentences

In [33]:
letters = df['s1_s2']
sentences = [sent_tokenize(letter) for letter in letters]
sentences = pd.concat([df, pd.Series(sentences, name='sentences')], axis=1)
sentences = sentences[['sentences']].explode('sentences')
df_sentences = pd.merge(df, sentences, left_index=True, right_index=True)

In [34]:
df_sentences

,s1_s2,full_text,label,sentences
0,it is my pleasure to write a letter of recomme...,it is my pleasure to write a letter of recomme...,0,it is my pleasure to write a letter of recomme...
0,it is my pleasure to write a letter of recomme...,it is my pleasure to write a letter of recomme...,0,their pleasing personality and sincere dedica...
0,it is my pleasure to write a letter of recomme...,it is my pleasure to write a letter of recomme...,0,i first met identifier during their hospic...
0,it is my pleasure to write a letter of recomme...,it is my pleasure to write a letter of recomme...,0,"as part of the team, identifier took part in..."
0,it is my pleasure to write a letter of recomme...,it is my pleasure to write a letter of recomme...,0,"prior to seeing patients, they diligently re..."
...,...,...,...,...
8986,please accept my strongest recommendation for ...,please accept my strongest recommendation for ...,1,they clearly is at the top of their medical...
8986,please accept my strongest recommendation for ...,please accept my strongest recommendation for ...,1,"furthermore , they has a strong research bac..."
8986,please accept my strongest recommendation for ...,please accept my strongest recommendation for ...,1,their work experience is rounded out with num...
8986,please accept my strongest recommendation for ...,please accept my strongest recommendation for ...,1,"taken together , identifier is identifier ..."


# Create Training and Test Sets

In [35]:
train_text, temp_text, train_labels, temp_labels = train_test_split(df_sentences['sentences'], df_sentences['label'],
                                                                    random_state=0,
                                                                    test_size=0.3,
                                                                    stratify=df_sentences['label'])


val_text, test_text, val_labels, test_labels = train_test_split(temp_text, temp_labels,
                                                                random_state=0,
                                                                test_size=0.5,
                                                                stratify=temp_labels)

# Random Undersampler

In [36]:
undersampler = RandomUnderSampler(random_state=0)
train_text, train_labels = undersampler.fit_resample(pd.DataFrame(train_text), train_labels)
train_text = train_text['sentences']

In [37]:
bert = AutoModel.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [38]:
tokens_train = tokenizer.batch_encode_plus(
    train_text.tolist(),
    padding='max_length',
    truncation=True
)

tokens_val = tokenizer.batch_encode_plus(
    val_text.tolist(),
    padding='max_length',
    truncation=True
)

tokens_test = tokenizer.batch_encode_plus(
    test_text.tolist(),
    padding='max_length',
    truncation=True
)

In [39]:
train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels.tolist())

In [40]:
batch_size = 8
train_data = TensorDataset(train_seq, train_mask, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_seq, val_mask, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [41]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


# Create Model

In [42]:
class BERT_Arch(nn.Module):

    def __init__(self, bert):

        super(BERT_Arch, self).__init__()

        self.bert = bert

        # dropout layer
        self.dropout = nn.Dropout(0.1)

        # relu activation function
        self.relu =  nn.ReLU()

        # dense layer 1
        self.fc1 = nn.Linear(768,512)

        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(512,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):

        #pass the inputs to the model
        distilbert_output = self.bert(sent_id, attention_mask=mask)

        hidden_state = distilbert_output[0]  # (bs, seq_len, dim)
        pooled_output = hidden_state[:, 0]
        # pooled_output = hidden_state.mean(dim=1)

        x = self.fc1(pooled_output)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)

        # apply softmax activation
        x = self.softmax(x)

        return x

In [43]:
model = BERT_Arch(bert)
model = model.to(device)

In [44]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss()
epochs = 5

In [45]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_dataloader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    # push the batch to gpu
    batch = [r.to(device) for r in batch]

    sent_id, mask, labels = batch

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(sent_id, mask)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_dataloader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [46]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_dataloader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    # push the batch to gpu
    batch = [t.to(device) for t in batch]

    sent_id, mask, labels = batch

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(sent_id, mask)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_dataloader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, total_preds

In [47]:
# set initial loss to infinite
best_valid_loss = float('inf')

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, _ = evaluate()

    #save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print('Model Saved!')
        torch.save(model, '../saved_models/saved_model_degendered_oversampled.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/5 [00:00<?, ?it/s]


 Epoch 1 / 5


Training:   0%|          | 0/8724 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.52      0.52      0.52     34894
           1       0.52      0.52      0.52     34894

    accuracy                           0.52     69788
   macro avg       0.52      0.52      0.52     69788
weighted avg       0.52      0.52      0.52     69788

Training Confusion Matrix: 
 [[18273 16621]
 [16770 18124]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.33      0.85      0.48      7477
           1       0.73      0.20      0.31     15799

    accuracy                           0.41     23276
   macro avg       0.53      0.52      0.40     23276
weighted avg       0.61      0.41      0.37     23276

Validation Confusion Matrix: 
 [[ 6334  1143]
 [12646  3153]]
Model Saved!

Training Loss: 0.693
Validation Loss: 0.718

 Epoch 2 / 5


Training:   0%|          | 0/8724 [00:00<?, ?it/s]

...

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.54      0.53      0.53     34894
           1       0.54      0.55      0.55     34894

    accuracy                           0.54     69788
   macro avg       0.54      0.54      0.54     69788
weighted avg       0.54      0.54      0.54     69788

Training Confusion Matrix: 
 [[18416 16478]
 [15604 19290]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.36      0.66      0.46      7477
           1       0.73      0.45      0.55     15799

    accuracy                           0.51     23276
   macro avg       0.55      0.55      0.51     23276
weighted avg       0.61      0.51      0.53     23276

Validation Confusion Matrix: 
 [[4915 2562]
 [8755 7044]]

Training Loss: 0.686
Validation Loss: 0.698

 Epoch 4 / 5


Training:   0%|          | 0/8724 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.55      0.54      0.55     34894
           1       0.55      0.55      0.55     34894

    accuracy                           0.55     69788
   macro avg       0.55      0.55      0.55     69788
weighted avg       0.55      0.55      0.55     69788

Training Confusion Matrix: 
 [[18961 15933]
 [15577 19317]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.44      0.19      0.27      7477
           1       0.70      0.88      0.78     15799

    accuracy                           0.66     23276
   macro avg       0.57      0.54      0.52     23276
weighted avg       0.62      0.66      0.62     23276

Validation Confusion Matrix: 
 [[ 1442  6035]
 [ 1838 13961]]
Model Saved!

Training Loss: 0.683
Validation Loss: 0.640

 Epoch 5 / 5


Training:   0%|          | 0/8724 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.55      0.54      0.55     34894
           1       0.55      0.56      0.56     34894

    accuracy                           0.55     69788
   macro avg       0.55      0.55      0.55     69788
weighted avg       0.55      0.55      0.55     69788

Training Confusion Matrix: 
 [[18879 16015]
 [15262 19632]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.36      0.74      0.48      7477
           1       0.75      0.37      0.49     15799

    accuracy                           0.49     23276
   macro avg       0.55      0.55      0.49     23276
weighted avg       0.62      0.49      0.49     23276

Validation Confusion Matrix: 
 [[5508 1969]
 [9969 5830]]

Training Loss: 0.680
Validation Loss: 0.703


# Apply on Test Set

In [48]:
model = torch.load("../saved_models/saved_model_degendered_oversampled.pt")
model.eval()

<ipython-input-48-f607ed88b2a0>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("../saved_models/saved_model_degendered_oversampled.pt")


BERT_Arch(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_fe

In [49]:
test_dataset = TensorDataset(test_seq, test_mask, test_y)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [50]:
all_preds = []
all_labels = []

with torch.no_grad():  # Disable gradient calculations for efficiency
    for batch in tqdm(test_loader, desc="Processing Batches", unit="batch"):
        batch_seq, batch_mask, batch_labels = batch  # Extract input tensors
        batch_seq, batch_mask = batch_seq.to(device), batch_mask.to(device)  # Move to device

        outputs = model(batch_seq, mask=batch_mask)  # Forward pass
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())  # Store predictions
        all_labels.extend(batch_labels.cpu().numpy())  # Store true labels


Processing Batches:   0%|          | 0/2910 [00:00<?, ?batch/s]

In [51]:
test_text = pd.DataFrame(test_text)
test_text["prediction"] = all_preds
test_text["true_label"] = all_labels

In [52]:
test_text.to_csv('../data/results/sentence_classifier_result_degendered_oversampled_data.csv', index=False)

In [53]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.44      0.19      0.26      7478
           1       0.70      0.89      0.78     15799

    accuracy                           0.66     23277
   macro avg       0.57      0.54      0.52     23277
weighted avg       0.61      0.66      0.61     23277

Test Confusion Matrix: 
 [[ 1403  6075]
 [ 1810 13989]]
